In [0]:
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_dna_member.generate_population as gp
import lib_dna_member.job_manager as job_manager
import lib_dna_member.misc_features as features
from lib_dna_member.s3 import member_dna_input_data_validator
from databricks.feature_engineering import FeatureEngineeringClient

In [0]:
%run ../../config/utils

In [0]:
def generate_misc(job):
    """
    Generate the miscellaneous variables for the given population
    Parameters:
        job (object): Job Manager object based on the current config file

    Returns:
        (pyspark.sql.DataFrame): Miscellaneous data of the given population
    """

    dna = job.tables["population"]
    
    member = job.tables["cubes_member_features"].select(
        "MBRSHP_SID",
        "FISCAL_WEEK_END",
        "LATEST_MBRSHP_NBR",
        "LATEST_HOME_ZIP_CD",
        "BJS_DISTANCE",
    )
    transaction1 = job.tables["cubes_transaction_1"].select(
        "MBRSHP_SID",
        "FISCAL_WEEK_END",
        "LAST_FIFTY-TWO_WEEK_TRIPS",
        "LAST_TWELVE_WEEK_TRIPS",
        "LAST_TWENTY-SIX_WEEK_TRIPS",
    )
    transaction2 = job.tables["cubes_transaction_2"].select(
        "MBRSHP_SID",
        "FISCAL_WEEK_END",
        "LAST_FIFTY-TWO_WEEK_SPEND",
        "LAST_TWELVE_WEEK_SPEND",
        "LAST_TWENTY-SIX_WEEK_SPEND",
    )

    dna = dna.join(
        member,
        ["MBRSHP_SID", "FISCAL_WEEK_END"],
        "left_outer"
    ).join(
        transaction1,
        ["MBRSHP_SID", "FISCAL_WEEK_END"],
        "left_outer"
    ).join(
        transaction2,
        ["MBRSHP_SID", "FISCAL_WEEK_END"],
        "left_outer"
    )
    orig_cols = dna.columns

    dna = features.feature_preferred_club(job, dna)
    dna = features.feature_dummy_member(job, dna)

    num_weeks = ["TWELVE", "TWENTY-SIX"]
    dna = features.feature_last_over_prior(job, dna, num_weeks, "spend")
    dna = features.feature_last_over_prior(job, dna, num_weeks, "trips")

    dna = features.feature_strategic_segment(job, dna)

    dna = features.feature_preferred_club_has_gas(job, dna, 52)

    dna = dna.drop(
        *[
            col
            for col in orig_cols
            if col not in ["MBRSHP_SID", "FISCAL_WEEK_END"]
        ]
    )

    return dna

In [0]:
job = job_manager.JobManager(spark, intermediate_all_tables_dict, member_dna_config_path)

In [0]:
recency_lookback_duration = job.config["params"].get(
    "recency_lookback_duration", {}
)

member_dna_input_data_validator(
    silver_transaction_fiscal_header, silver_transaction_fiscal_detail_isnr, silver_skeleton, silver_bcg_maps_strategic_segments, silver_master_member_extended,  fs_cubes_transaction_1, fs_cubes_transaction_2, fs_cubes_member, silver_master_club_with_brand,
    recency_lookback_duration=recency_lookback_duration,
    spark=spark
)

In [0]:
job.read_table("header_fiscal")
job.read_table("detail_isnr_fiscal")
job.read_table("skeleton")
job.read_table("member_extended")
job.read_table("segment")
job.read_table("cubes_transaction_1")
job.read_table("cubes_transaction_2")
job.read_table("cubes_member_features")
job.read_table("club")

In [0]:
job.tables["header_fiscal"] = gp.apply_fw_date_range(
    job, job.tables["header_fiscal"]
)

job.tables["detail_isnr_fiscal"] = gp.apply_fw_date_range(
    job, job.tables["detail_isnr_fiscal"]
)

population = gp.generate_population(job)
job.tables["population"] = population

feature_population = gp.generate_population(job, "feature")
job.tables["feature_population"] = feature_population

misc_features = generate_misc(job)
#job.tables["cubes_misc"] = misc_features

### Save results

In [0]:
#### IMPORTANT     This was NOT in the original code but the target table has a NOT NULL constrain on this column

misc_features = misc_features.filter("MBRSHP_SID IS NOT NULL")

In [0]:
spark.sql(f"DELETE FROM {fs_cubes_misc}")

fe = FeatureEngineeringClient()

fe.write_table(
    name=fs_cubes_misc,  
    df=misc_features,
    mode="merge"
)